
# Buy Fee Model V10 — Success vs. Failure Diagnostics

## Objective

V10 is a **diagnostic notebook**, not a final pricing recommendation model.

Its job is to answer:

> **What is different about Buy Fee changes that worked well versus those that performed poorly?**

The analysis compares successful versus unsuccessful historical Buy Fee changes by:

- Location
- Price tier
- Lane
- Fee-change dollars
- Fee-change percentage
- Baseline conversion
- Baseline volume
- Previous Buy Fee
- Baseline RPU
- Seasonality
- Buyer concentration (optional enrichment)
- Corporate change date
- Historical support

---

## Source

V10 starts from the **V8.2 matched-control Difference-in-Differences event-level dataset**:

`04_DID_EVENT_LEVEL_RESULTS.csv`

This means the outcomes are already adjusted against matched controls from the same calendar periods.

---

## Success definitions

A single definition of “successful” can hide tradeoffs, so V10 creates multiple views.

### RPU Success
A fee change is considered RPU-successful when:

`DiD_RPU_Change_Dollar > 0`

### Conversion Safe
A fee change is considered conversion-safe when:

`DiD_Conversion_Change_pp >= -2`

### Volume Safe
A fee change is considered volume-safe when:

`DiD_Volume_Change_Pct >= -10%`

### Balanced Success
A fee change is a **Balanced Success** when:

- RPU gain > 0
- Conversion decline is no worse than 2 percentage points
- Volume decline is no worse than 10%

This is a diagnostic definition only. The thresholds can be changed in the configuration section.

---

## Buyer concentration

Buyer concentration is optional.

If you have a separate buyer file, V10 can join a pre-aggregated buyer concentration table with fields such as:

- VRLOCATION
- Event_Week / Week_Start
- price_tier
- Top_Buyer_Share
- Top5_Buyer_Share
- Unique_Buyers
- HHI / concentration metric

The notebook runs fully without it and clearly reports when buyer concentration is unavailable.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import warnings

warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v10_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# OPTIONAL BUYER CONCENTRATION INPUT
# ------------------------------------------------------------
# Leave as None if not available yet.
#
# Recommended grain:
# VRLOCATION x Week_Start x price_tier
#
# Expected optional fields:
# Top_Buyer_Share
# Top5_Buyer_Share
# Unique_Buyers
# Buyer_HHI

BUYER_CONCENTRATION_FILE = None
# Example:
# BUYER_CONCENTRATION_FILE = DATA_FOLDER / "BUYER_CONCENTRATION_WEEKLY.csv"

# ------------------------------------------------------------
# SUCCESS THRESHOLDS
# ------------------------------------------------------------

RPU_SUCCESS_MIN = 0.0

CONVERSION_SAFE_MIN_PP = -2.0

VOLUME_SAFE_MIN_PCT = -0.10

# Minimum support for group-level reporting
MIN_GROUP_EVENTS = 10

# Bin counts
N_QUANTILE_BINS = 4

print("=" * 80)
print("V10 INPUT CHECK")
print("=" * 80)

print("DiD file:", DID_FILE)
print("Exists:", DID_FILE.exists())

if not DID_FILE.exists():
    raise FileNotFoundError(DID_FILE)

print("\nBuyer file configured:", BUYER_CONCENTRATION_FILE)


## 1. Load matched-control DiD events

In [ ]:

df = pd.read_csv(
    DID_FILE,
    low_memory=True
)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))

required = [
    "Segment_Change_ID",
    "Corporate_Change_ID",
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "Treated_Location",
    "Treated_Lane",
    "Treated_Price_Tier",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

missing = [
    c for c in required
    if c not in df.columns
]

if missing:
    raise ValueError(
        "Missing required V8.2 columns: "
        + ", ".join(missing)
    )

numeric_cols = [
    "BUYFEE_CHANGE_AMOUNT",
    "PREVIOUS_BUYFEE",
    "FEE_CHANGE_PCT",
    "DiD_Volume_Change_Pct",
    "DiD_Conversion_Change_pp",
    "DiD_RPU_Change_Dollar",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

for c in numeric_cols:
    df[c] = pd.to_numeric(
        df[c],
        errors="coerce"
    )

print("\nUnique locations:", df["Treated_Location"].nunique())
print("Unique lanes:", df["Treated_Lane"].nunique())
print("Corporate change dates:", df["Corporate_Change_ID"].nunique())

display(df.head())


## 2. Derive event date and seasonality

In [ ]:

# Corporate_Change_ID was created as YYYY-MM-DD in the prior pipeline.
df["Change_Date"] = pd.to_datetime(
    df["Corporate_Change_ID"],
    errors="coerce"
)

df["Change_Year"] = df["Change_Date"].dt.year
df["Change_Month"] = df["Change_Date"].dt.month
df["Change_Month_Name"] = df["Change_Date"].dt.month_name()
df["Change_Quarter"] = df["Change_Date"].dt.quarter
df["Change_Week_of_Year"] = (
    df["Change_Date"]
    .dt.isocalendar()
    .week
    .astype("Int64")
)

def season_from_month(m):
    if pd.isna(m):
        return "Unknown"

    if m in [12, 1, 2]:
        return "Winter"

    if m in [3, 4, 5]:
        return "Spring"

    if m in [6, 7, 8]:
        return "Summer"

    return "Fall"

df["Season"] = df["Change_Month"].apply(
    season_from_month
)

display(
    df[
        [
            "Corporate_Change_ID",
            "Change_Date",
            "Change_Month_Name",
            "Change_Quarter",
            "Season"
        ]
    ].drop_duplicates().head(20)
)


## 3. Create success / failure flags

In [ ]:

df["RPU_Success"] = (
    df["DiD_RPU_Change_Dollar"] > RPU_SUCCESS_MIN
)

df["Conversion_Safe"] = (
    df["DiD_Conversion_Change_pp"] >= CONVERSION_SAFE_MIN_PP
)

df["Volume_Safe"] = (
    df["DiD_Volume_Change_Pct"] >= VOLUME_SAFE_MIN_PCT
)

df["Balanced_Success"] = (
    df["RPU_Success"]
    &
    df["Conversion_Safe"]
    &
    df["Volume_Safe"]
)

df["Outcome_Class"] = np.select(
    [
        df["Balanced_Success"],
        df["RPU_Success"] & ~df["Conversion_Safe"],
        df["RPU_Success"] & ~df["Volume_Safe"],
        ~df["RPU_Success"] & df["Conversion_Safe"] & df["Volume_Safe"],
    ],
    [
        "Balanced Success",
        "RPU Gain / Conversion Risk",
        "RPU Gain / Volume Risk",
        "No RPU Gain / Operationally Safe",
    ],
    default="Unsuccessful"
)

summary = pd.DataFrame([
    {
        "Metric": "Events",
        "Value": len(df)
    },
    {
        "Metric": "RPU_Success_Rate",
        "Value": df["RPU_Success"].mean()
    },
    {
        "Metric": "Conversion_Safe_Rate",
        "Value": df["Conversion_Safe"].mean()
    },
    {
        "Metric": "Volume_Safe_Rate",
        "Value": df["Volume_Safe"].mean()
    },
    {
        "Metric": "Balanced_Success_Rate",
        "Value": df["Balanced_Success"].mean()
    },
])

display(summary)

display(
    df["Outcome_Class"]
    .value_counts(dropna=False)
    .rename_axis("Outcome_Class")
    .reset_index(name="Events")
)


## 4. Optional buyer-concentration enrichment

In [ ]:

BUYER_DATA_AVAILABLE = False

if BUYER_CONCENTRATION_FILE is not None:

    buyer_path = Path(
        BUYER_CONCENTRATION_FILE
    )

    if buyer_path.exists():

        buyer = pd.read_csv(
            buyer_path,
            low_memory=True
        )

        print("Buyer file loaded:", buyer_path)
        print("Buyer rows:", f"{len(buyer):,}")

        buyer_cols = buyer.columns.tolist()

        print("Buyer columns:")
        print(buyer_cols)

        # The notebook intentionally does not guess a complex buyer join.
        # If a clean event-level Buyer_Concentration field already exists,
        # it will join directly by Segment_Change_ID.

        if "Segment_Change_ID" in buyer.columns:

            buyer_keep = [
                c for c in [
                    "Segment_Change_ID",
                    "Top_Buyer_Share",
                    "Top5_Buyer_Share",
                    "Unique_Buyers",
                    "Buyer_HHI"
                ]
                if c in buyer.columns
            ]

            buyer_small = (
                buyer[buyer_keep]
                .drop_duplicates(
                    subset=["Segment_Change_ID"]
                )
            )

            df = df.merge(
                buyer_small,
                on="Segment_Change_ID",
                how="left"
            )

            BUYER_DATA_AVAILABLE = True

        else:

            print(
                "\nBuyer file does not contain Segment_Change_ID."
            )

            print(
                "V10 will continue without buyer concentration."
            )

    else:

        print(
            "Configured buyer file was not found."
        )

else:

    print(
        "No buyer concentration file configured."
    )

print(
    "\nBuyer enrichment available:",
    BUYER_DATA_AVAILABLE
)


## 5. Create diagnostic bins

In [ ]:

def safe_qcut(series, q=4, prefix="Q"):
    s = pd.to_numeric(
        series,
        errors="coerce"
    )

    try:
        return pd.qcut(
            s,
            q=q,
            duplicates="drop"
        )
    except Exception:
        return pd.Series(
            "Not Available",
            index=series.index
        )

df["Fee_Change_Pct_Band"] = safe_qcut(
    df["FEE_CHANGE_PCT"],
    q=N_QUANTILE_BINS
)

df["Baseline_Conversion_Band"] = safe_qcut(
    df["pre_conversion"],
    q=N_QUANTILE_BINS
)

df["Baseline_Volume_Band"] = safe_qcut(
    df["pre_sold"],
    q=N_QUANTILE_BINS
)

df["Previous_BuyFee_Band"] = safe_qcut(
    df["PREVIOUS_BUYFEE"],
    q=N_QUANTILE_BINS
)

df["Baseline_RPU_Band"] = safe_qcut(
    df["pre_rpu"],
    q=N_QUANTILE_BINS
)

df["Sale_Amount_Band"] = safe_qcut(
    df["pre_avg_saleamt"],
    q=N_QUANTILE_BINS
)

if BUYER_DATA_AVAILABLE:

    for col in [
        "Top_Buyer_Share",
        "Top5_Buyer_Share",
        "Unique_Buyers",
        "Buyer_HHI"
    ]:

        if col in df.columns:

            df[
                f"{col}_Band"
            ] = safe_qcut(
                df[col],
                q=N_QUANTILE_BINS
            )


## 6. Core comparison function

In [ ]:

def compare_groups(
    data,
    group_col,
    min_events=10
):

    temp = data[
        data[group_col].notna()
    ].copy()

    result = (
        temp.groupby(
            group_col,
            dropna=False
        )
        .agg(
            Events=("Segment_Change_ID", "count"),

            Locations=(
                "Treated_Location",
                "nunique"
            ),

            Change_Dates=(
                "Corporate_Change_ID",
                "nunique"
            ),

            Balanced_Success_Rate=(
                "Balanced_Success",
                "mean"
            ),

            RPU_Success_Rate=(
                "RPU_Success",
                "mean"
            ),

            Conversion_Safe_Rate=(
                "Conversion_Safe",
                "mean"
            ),

            Volume_Safe_Rate=(
                "Volume_Safe",
                "mean"
            ),

            Median_RPU_Effect=(
                "DiD_RPU_Change_Dollar",
                "median"
            ),

            Median_Conversion_Effect_pp=(
                "DiD_Conversion_Change_pp",
                "median"
            ),

            Median_Volume_Effect_Pct=(
                "DiD_Volume_Change_Pct",
                "median"
            ),

            Median_Fee_Change=(
                "BUYFEE_CHANGE_AMOUNT",
                "median"
            ),

            Median_Fee_Change_Pct=(
                "FEE_CHANGE_PCT",
                "median"
            ),

            Median_Baseline_Conversion=(
                "pre_conversion",
                "median"
            ),

            Median_Baseline_Sold=(
                "pre_sold",
                "median"
            ),

            Median_Previous_BuyFee=(
                "PREVIOUS_BUYFEE",
                "median"
            ),
        )
        .reset_index()
    )

    result = result[
        result["Events"] >= min_events
    ].copy()

    result = result.sort_values(
        [
            "Balanced_Success_Rate",
            "Events"
        ],
        ascending=[
            False,
            False
        ]
    )

    return result


## 7. Compare by Location

In [ ]:

location_comparison = compare_groups(
    df,
    "Treated_Location",
    min_events=MIN_GROUP_EVENTS
)

location_comparison.to_csv(
    OUTPUT_DIR / "01_LOCATION_SUCCESS_DIAGNOSTICS.csv",
    index=False
)

display(
    location_comparison.head(25)
)


## 8. Compare by Price Tier

In [ ]:

price_tier_comparison = compare_groups(
    df,
    "Treated_Price_Tier",
    min_events=MIN_GROUP_EVENTS
)

price_tier_comparison.to_csv(
    OUTPUT_DIR / "02_PRICE_TIER_SUCCESS_DIAGNOSTICS.csv",
    index=False
)

display(
    price_tier_comparison.head(30)
)


## 9. Compare by Lane

In [ ]:

lane_comparison = compare_groups(
    df,
    "Treated_Lane",
    min_events=MIN_GROUP_EVENTS
)

lane_comparison.to_csv(
    OUTPUT_DIR / "03_LANE_SUCCESS_DIAGNOSTICS.csv",
    index=False
)

display(
    lane_comparison.head(30)
)


## 10. Compare by Fee-Change Percentage

In [ ]:

fee_pct_comparison = compare_groups(
    df,
    "Fee_Change_Pct_Band",
    min_events=MIN_GROUP_EVENTS
)

fee_pct_comparison.to_csv(
    OUTPUT_DIR / "04_FEE_CHANGE_PCT_DIAGNOSTICS.csv",
    index=False
)

display(fee_pct_comparison)


## 11. Compare by baseline Conversion

In [ ]:

baseline_conversion = compare_groups(
    df,
    "Baseline_Conversion_Band",
    min_events=MIN_GROUP_EVENTS
)

baseline_conversion.to_csv(
    OUTPUT_DIR / "05_BASELINE_CONVERSION_DIAGNOSTICS.csv",
    index=False
)

display(baseline_conversion)


## 12. Compare by baseline Volume

In [ ]:

baseline_volume = compare_groups(
    df,
    "Baseline_Volume_Band",
    min_events=MIN_GROUP_EVENTS
)

baseline_volume.to_csv(
    OUTPUT_DIR / "06_BASELINE_VOLUME_DIAGNOSTICS.csv",
    index=False
)

display(baseline_volume)


## 13. Compare by Seasonality

In [ ]:

season_comparison = compare_groups(
    df,
    "Season",
    min_events=MIN_GROUP_EVENTS
)

quarter_comparison = compare_groups(
    df,
    "Change_Quarter",
    min_events=MIN_GROUP_EVENTS
)

month_comparison = compare_groups(
    df,
    "Change_Month_Name",
    min_events=MIN_GROUP_EVENTS
)

season_comparison.to_csv(
    OUTPUT_DIR / "07_SEASON_DIAGNOSTICS.csv",
    index=False
)

quarter_comparison.to_csv(
    OUTPUT_DIR / "08_QUARTER_DIAGNOSTICS.csv",
    index=False
)

month_comparison.to_csv(
    OUTPUT_DIR / "09_MONTH_DIAGNOSTICS.csv",
    index=False
)

display(season_comparison)
display(quarter_comparison)


## 14. Compare by buyer concentration (optional)

In [ ]:

buyer_outputs = []

if BUYER_DATA_AVAILABLE:

    for col in [
        "Top_Buyer_Share",
        "Top5_Buyer_Share",
        "Unique_Buyers",
        "Buyer_HHI"
    ]:

        band_col = f"{col}_Band"

        if band_col not in df.columns:
            continue

        result = compare_groups(
            df,
            band_col,
            min_events=MIN_GROUP_EVENTS
        )

        path = (
            OUTPUT_DIR
            / f"BUYER_{col.upper()}_DIAGNOSTICS.csv"
        )

        result.to_csv(
            path,
            index=False
        )

        buyer_outputs.append(
            str(path)
        )

        print("\n", col)
        display(result)

else:

    print(
        "Buyer concentration was not available."
    )

    print(
        "Core V10 diagnostics are still complete."
    )


## 15. Successful vs unsuccessful event profiles

In [ ]:

profile_features = [
    "BUYFEE_CHANGE_AMOUNT",
    "FEE_CHANGE_PCT",
    "PREVIOUS_BUYFEE",
    "pre_registered",
    "pre_sold",
    "pre_conversion",
    "pre_rpu",
    "pre_avg_saleamt",
]

if BUYER_DATA_AVAILABLE:

    profile_features += [
        c for c in [
            "Top_Buyer_Share",
            "Top5_Buyer_Share",
            "Unique_Buyers",
            "Buyer_HHI"
        ]
        if c in df.columns
    ]

profile_rows = []

for feature in profile_features:

    x = pd.to_numeric(
        df[feature],
        errors="coerce"
    )

    success = x[
        df["Balanced_Success"]
    ].dropna()

    fail = x[
        ~df["Balanced_Success"]
    ].dropna()

    profile_rows.append({
        "Feature": feature,

        "Successful_N": len(success),
        "Unsuccessful_N": len(fail),

        "Successful_Median": (
            success.median()
            if len(success)
            else np.nan
        ),

        "Unsuccessful_Median": (
            fail.median()
            if len(fail)
            else np.nan
        ),

        "Median_Difference": (
            success.median()
            - fail.median()
            if len(success)
            and len(fail)
            else np.nan
        ),

        "Successful_Mean": (
            success.mean()
            if len(success)
            else np.nan
        ),

        "Unsuccessful_Mean": (
            fail.mean()
            if len(fail)
            else np.nan
        )
    })

success_failure_profile = pd.DataFrame(
    profile_rows
)

success_failure_profile.to_csv(
    OUTPUT_DIR / "10_SUCCESS_VS_FAILURE_PROFILE.csv",
    index=False
)

display(success_failure_profile)


## 16. Rank diagnostic drivers by spread in success rate

In [ ]:

driver_tables = {
    "Location": location_comparison,
    "Price Tier": price_tier_comparison,
    "Lane": lane_comparison,
    "Fee Change %": fee_pct_comparison,
    "Baseline Conversion": baseline_conversion,
    "Baseline Volume": baseline_volume,
    "Season": season_comparison,
    "Quarter": quarter_comparison,
}

driver_rows = []

for driver, table in driver_tables.items():

    if table.empty:
        continue

    max_rate = table[
        "Balanced_Success_Rate"
    ].max()

    min_rate = table[
        "Balanced_Success_Rate"
    ].min()

    driver_rows.append({
        "Driver": driver,
        "Groups_Compared": len(table),
        "Best_Success_Rate": max_rate,
        "Worst_Success_Rate": min_rate,
        "Success_Rate_Spread": (
            max_rate - min_rate
        ),
        "Events_Represented": table[
            "Events"
        ].sum()
    })

driver_importance = (
    pd.DataFrame(driver_rows)
    .sort_values(
        "Success_Rate_Spread",
        ascending=False
    )
)

driver_importance.to_csv(
    OUTPUT_DIR / "11_DRIVER_DIAGNOSTIC_RANKING.csv",
    index=False
)

display(driver_importance)


## 17. Scenario-specific successful vs unsuccessful diagnostics

In [ ]:

scenario_rows = []

for fee in [10, 15, 20, 25, 30]:

    d = df[
        df["BUYFEE_CHANGE_AMOUNT"].between(
            fee - 2.5,
            fee + 2.5
        )
    ].copy()

    if d.empty:
        continue

    success = d[
        d["Balanced_Success"]
    ]

    fail = d[
        ~d["Balanced_Success"]
    ]

    scenario_rows.append({
        "Fee_Increase": fee,

        "Events": len(d),
        "Locations": d[
            "Treated_Location"
        ].nunique(),

        "Corporate_Change_Dates": d[
            "Corporate_Change_ID"
        ].nunique(),

        "Balanced_Success_Rate": d[
            "Balanced_Success"
        ].mean(),

        "RPU_Success_Rate": d[
            "RPU_Success"
        ].mean(),

        "Conversion_Safe_Rate": d[
            "Conversion_Safe"
        ].mean(),

        "Volume_Safe_Rate": d[
            "Volume_Safe"
        ].mean(),

        "Successful_Median_Baseline_Conversion": (
            success["pre_conversion"].median()
            if len(success)
            else np.nan
        ),

        "Unsuccessful_Median_Baseline_Conversion": (
            fail["pre_conversion"].median()
            if len(fail)
            else np.nan
        ),

        "Successful_Median_Baseline_Volume": (
            success["pre_sold"].median()
            if len(success)
            else np.nan
        ),

        "Unsuccessful_Median_Baseline_Volume": (
            fail["pre_sold"].median()
            if len(fail)
            else np.nan
        ),

        "Successful_Median_Previous_BuyFee": (
            success["PREVIOUS_BUYFEE"].median()
            if len(success)
            else np.nan
        ),

        "Unsuccessful_Median_Previous_BuyFee": (
            fail["PREVIOUS_BUYFEE"].median()
            if len(fail)
            else np.nan
        ),
    })

scenario_diagnostics = pd.DataFrame(
    scenario_rows
)

scenario_diagnostics.to_csv(
    OUTPUT_DIR / "12_SCENARIO_SUCCESS_DIAGNOSTICS.csv",
    index=False
)

display(scenario_diagnostics)


## 18. Simple diagnostic charts

In [ ]:

def plot_success_table(
    table,
    group_col,
    title,
    filename,
    top_n=15
):

    if table.empty:
        return

    d = table.head(top_n).copy()

    labels = (
        d[group_col]
        .astype(str)
    )

    values = (
        d["Balanced_Success_Rate"]
        * 100
    )

    plt.figure(
        figsize=(10, 6)
    )

    plt.barh(
        labels[::-1],
        values[::-1]
    )

    plt.xlabel(
        "Balanced Success Rate (%)"
    )

    plt.title(
        title
    )

    plt.tight_layout()

    plt.savefig(
        OUTPUT_DIR / filename,
        dpi=150,
        bbox_inches="tight"
    )

    plt.show()


plot_success_table(
    location_comparison,
    "Treated_Location",
    "Balanced Buy Fee Success by Location",
    "13_SUCCESS_BY_LOCATION.png",
    top_n=15
)

plot_success_table(
    price_tier_comparison,
    "Treated_Price_Tier",
    "Balanced Buy Fee Success by Price Tier",
    "14_SUCCESS_BY_PRICE_TIER.png",
    top_n=15
)

plot_success_table(
    lane_comparison,
    "Treated_Lane",
    "Balanced Buy Fee Success by Lane",
    "15_SUCCESS_BY_LANE.png",
    top_n=15
)


## 19. Final V10 executive diagnostic summary

In [ ]:

overall = {
    "Events": len(df),

    "Locations": df[
        "Treated_Location"
    ].nunique(),

    "Corporate_Change_Dates": df[
        "Corporate_Change_ID"
    ].nunique(),

    "Balanced_Success_Rate": df[
        "Balanced_Success"
    ].mean(),

    "RPU_Success_Rate": df[
        "RPU_Success"
    ].mean(),

    "Conversion_Safe_Rate": df[
        "Conversion_Safe"
    ].mean(),

    "Volume_Safe_Rate": df[
        "Volume_Safe"
    ].mean(),

    "Buyer_Data_Available": (
        BUYER_DATA_AVAILABLE
    )
}

executive_rows = [
    {
        "Section": "Overall",
        "Finding": k,
        "Value": v
    }
    for k, v in overall.items()
]

if not driver_importance.empty:

    for _, r in driver_importance.head(5).iterrows():

        executive_rows.append({
            "Section": "Driver Ranking",
            "Finding": r["Driver"],
            "Value": r[
                "Success_Rate_Spread"
            ]
        })

executive_summary = pd.DataFrame(
    executive_rows
)

executive_summary.to_csv(
    OUTPUT_DIR / "16_V10_EXECUTIVE_DIAGNOSTIC_SUMMARY.csv",
    index=False
)

df.to_csv(
    OUTPUT_DIR / "17_V10_EVENT_LEVEL_ENRICHED.csv",
    index=False
)

print("=" * 80)
print("V10 COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(executive_summary)



## Files to share back

Please send these first:

1. `16_V10_EXECUTIVE_DIAGNOSTIC_SUMMARY.csv`
2. `11_DRIVER_DIAGNOSTIC_RANKING.csv`
3. `10_SUCCESS_VS_FAILURE_PROFILE.csv`
4. `12_SCENARIO_SUCCESS_DIAGNOSTICS.csv`
5. `01_LOCATION_SUCCESS_DIAGNOSTICS.csv`
6. `02_PRICE_TIER_SUCCESS_DIAGNOSTICS.csv`
7. `04_FEE_CHANGE_PCT_DIAGNOSTICS.csv`
8. `05_BASELINE_CONVERSION_DIAGNOSTICS.csv`
9. `06_BASELINE_VOLUME_DIAGNOSTICS.csv`
10. `07_SEASON_DIAGNOSTICS.csv`

If buyer concentration is added, also share the generated `BUYER_..._DIAGNOSTICS.csv` files.

## What comes after V10

V10 should tell us whether the strongest differences between successful and unsuccessful fee changes come from:

- price tier,
- baseline conversion,
- baseline volume,
- location,
- fee-change percentage,
- seasonality,
- or buyer concentration.

Those findings determine the features that should go into the final predictive model rather than adding variables blindly.
